# 🔍 Silent Failure Predictor — Exploratory Notebook

This notebook walks through the complete pipeline interactively:
1. Dataset generation & exploration
2. Feature engineering
3. Isolation Forest training & evaluation
4. Autoencoder training & reconstruction error analysis
5. Rich visualisations

**Run cells top-to-bottom.** Make sure you have installed all dependencies first:
```
pip install -r requirements.txt
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Project imports
from src.data_generator import generate_dataset
from src.preprocessor import prepare_data
from src.isolation_forest_model import IsolationForestDetector
from src.visualizer import (
    plot_metrics_overview,
    plot_anomaly_score_distribution,
    plot_confusion_matrix,
    build_plotly_figure,
)

print('✅ Imports OK')

## 1. Generate & Explore the Dataset

In [ ]:
df = generate_dataset(save=True)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Basic Statistics ===')
print(df.describe().round(2))

print('\n=== Class Balance ===')
print(df['is_anomaly'].value_counts())
print(f"Anomaly rate: {df['is_anomaly'].mean()*100:.2f}%")

In [ ]:
# Correlation heatmap
import seaborn as sns

numeric_cols = ['cpu_usage','memory_usage','response_time_ms',
                'disk_io_mbps','network_latency_ms','error_rate_pct']

fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Metrics Overview Plot

In [ ]:
path = plot_metrics_overview(df, filename='nb_metrics_overview.png')
from IPython.display import Image
Image(path)

In [ ]:
# Interactive Plotly chart for CPU usage
fig = build_plotly_figure(df, metric='cpu_usage')
fig.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
X, y, preprocessor = prepare_data(df)

print(f'Feature matrix shape : {X.shape}')
print(f'Feature names        : {preprocessor.feature_names}')
print(f'Labels shape         : {y.shape}')
print(f'Anomaly count        : {y.sum()} / {len(y)}')

## 4. Isolation Forest — Training & Evaluation

**Why Isolation Forest?**
- Unsupervised — no need for labelled anomaly data in production
- Efficient O(n log n) — suitable for high-volume metric streams
- Interpretable — anomaly score is the average path length in the forest

In [ ]:
detector = IsolationForestDetector()
detector.train(X)

labels, scores = detector.predict(X)
metrics        = detector.evaluate(X, y)

print(f"ROC-AUC          : {metrics['roc_auc']}")
print(f"Average Precision: {metrics['average_precision']}")

In [ ]:
# Anomaly score distribution
from IPython.display import Image
path = plot_anomaly_score_distribution(scores, y, filename='nb_score_dist.png')
Image(path)

In [ ]:
# Confusion matrix
path = plot_confusion_matrix(metrics['confusion_matrix'], filename='nb_confusion.png')
Image(path)

In [ ]:
# Anomaly scores over time — interactive
df_results = df.copy()
df_results['anomaly_score'] = scores
df_results['if_label']      = labels

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_results['timestamp'], y=df_results['anomaly_score'],
    mode='lines', name='Anomaly Score',
    line=dict(color='#4B9CFF', width=1)
))
anom_df = df_results[df_results['if_label']==1]
fig.add_trace(go.Scatter(
    x=anom_df['timestamp'], y=anom_df['anomaly_score'],
    mode='markers', name='Detected Anomaly',
    marker=dict(color='#FF4B4B', size=8, symbol='x')
))
fig.update_layout(
    title='Anomaly Score Over Time',
    template='plotly_dark',
    xaxis_title='Time', yaxis_title='Anomaly Score',
    height=400
)
fig.show()

## 5. Autoencoder — Bonus Model

**How it works:** Trained exclusively on normal samples, the autoencoder learns to reconstruct normal behaviour. Anomalies produce high reconstruction error.

In [ ]:
try:
    from src.autoencoder_model import AutoencoderDetector
    from src.visualizer import plot_reconstruction_error
    from sklearn.metrics import classification_report, roc_auc_score

    # Train only on normal data
    X_normal = X[y == 0]
    ae = AutoencoderDetector(input_dim=X.shape[1])
    ae.train(X_normal)

    ae_labels, ae_errors = ae.predict(X)
    ae_roc = roc_auc_score(y, ae_errors)

    print(f'Autoencoder ROC-AUC: {ae_roc:.4f}')
    print(f'Threshold          : {ae.threshold:.6f}')
    print()
    print(classification_report(y, ae_labels, target_names=['Normal','Anomaly']))

except ImportError:
    print('TensorFlow not installed. Skipping Autoencoder section.')
    print('Install with: pip install tensorflow')

In [ ]:
try:
    path = plot_reconstruction_error(ae_errors, ae.threshold, y, filename='nb_recon_error.png')
    Image(path)
except NameError:
    print('Autoencoder not trained — skipping plot.')

## 6. Model Comparison

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

results = {
    'Isolation Forest': {
        'ROC-AUC'          : metrics['roc_auc'],
        'Avg Precision'    : metrics['average_precision'],
        'Anomalies Found'  : int(labels.sum()),
    }
}

try:
    results['Autoencoder'] = {
        'ROC-AUC'         : round(ae_roc, 4),
        'Avg Precision'   : round(average_precision_score(y, ae_errors), 4),
        'Anomalies Found' : int(ae_labels.sum()),
    }
except NameError:
    pass

pd.DataFrame(results).T

## 7. Save Models for API & Dashboard

In [ ]:
import os; os.makedirs('../models', exist_ok=True)
detector.save('../models/isolation_forest.pkl')
preprocessor.save('../models/scaler.pkl')

try:
    ae.save('../models/autoencoder.h5')
except NameError:
    pass

print('✅ All models saved to models/')